# Pilot C Eval / Inference

Evaluate every 100-step checkpoint quickly, run precise conditional sequential evaluation for top checkpoints, save the best adapter, and generate test submission.

In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_pilot_c_deps_installed")

if Path("/content").exists() and not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "huggingface_hub",
        "hf_xet",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
elif Path("/content").exists():
    print("Dependencies already installed. Continue.")
else:
    print("Local environment detected. Skipping Colab dependency install.")


In [ ]:
# 2) Setup: Drive, data, model cache, run paths
from google.colab import drive
from pathlib import Path

drive_root = Path("/content/drive")
if drive_root.exists() and not os.path.ismount(str(drive_root)) and any(drive_root.iterdir()):
    import shutil
    print("Removing local pre-mount /content/drive contents:", sorted(str(p) for p in drive_root.iterdir())[:20])
    shutil.rmtree(drive_root)
drive_root.mkdir(parents=True, exist_ok=True)
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import itertools
import json
import math
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
import random
import re
import shutil
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, TrainerCallback, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

SNU_ROOT = Path("/content/drive/MyDrive/SNU_AI_Challenge")
assert SNU_ROOT.exists(), SNU_ROOT

ZIP_PATH = SNU_ROOT / "snuaichallenge.zip"
DATA_DIR = Path("/content/snuaichallenge_data")
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
TRAIN_IMAGE_DIR = DATA_DIR / "train"
TEST_IMAGE_DIR = DATA_DIR / "test"

if not TRAIN_CSV.exists() or not TRAIN_IMAGE_DIR.is_dir():
    print("Extracting:", ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert TRAIN_CSV.exists(), TRAIN_CSV
assert TEST_CSV.exists(), TEST_CSV
assert TRAIN_IMAGE_DIR.is_dir(), TRAIN_IMAGE_DIR
assert TEST_IMAGE_DIR.is_dir(), TEST_IMAGE_DIR

MODEL_REPO_ID = "Qwen/Qwen2-VL-7B-Instruct"
USE_MODELSCOPE_BASE_MODEL = False
DRIVE_MODEL_DIR = SNU_ROOT / "model_cache/Qwen2-VL-7B-Instruct"


def model_cache_is_complete(model_dir):
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").exists():
        return False
    has_weight = (model_dir / "model.safetensors.index.json").exists() or bool(list(model_dir.glob("*.safetensors")))
    has_processor = any((model_dir / name).exists() for name in ["preprocessor_config.json", "processor_config.json", "tokenizer.json", "tokenizer_config.json"])
    return bool(has_weight and has_processor)


def ensure_base_model_path():
    if model_cache_is_complete(DRIVE_MODEL_DIR):
        print("Using cached base model:", DRIVE_MODEL_DIR)
        return str(DRIVE_MODEL_DIR)
    if not USE_MODELSCOPE_BASE_MODEL:
        print("Base model cache not found. Downloading with Hugging Face snapshot_download to local disk first:")
        print("repo:", MODEL_REPO_ID)
        local_model_dir = Path("/content/Qwen2-VL-7B-Instruct")
        if local_model_dir.exists() and not model_cache_is_complete(local_model_dir):
            shutil.rmtree(local_model_dir)
        from huggingface_hub import snapshot_download
        hf_token = os.environ.get("HF_TOKEN")
        model_dir = snapshot_download(
            repo_id=MODEL_REPO_ID,
            local_dir=str(local_model_dir),
            token=hf_token,
            max_workers=8,
        )
        print("Downloaded:", model_dir)
        DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
        tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(local_model_dir, tmp, dirs_exist_ok=True)
        if DRIVE_MODEL_DIR.exists():
            shutil.rmtree(DRIVE_MODEL_DIR)
        os.replace(tmp, DRIVE_MODEL_DIR)
        print("Base model cached at:", DRIVE_MODEL_DIR)
        return str(DRIVE_MODEL_DIR)

    print("Base model cache not found. Downloading via ModelScope:", MODEL_REPO_ID)
    from modelscope import snapshot_download as modelscope_snapshot_download
    model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir="/content/modelscope_cache")
    DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    shutil.copytree(model_dir, tmp, dirs_exist_ok=True)
    if DRIVE_MODEL_DIR.exists():
        shutil.rmtree(DRIVE_MODEL_DIR)
    os.replace(tmp, DRIVE_MODEL_DIR)
    print("Base model cached at:", DRIVE_MODEL_DIR)
    return str(DRIVE_MODEL_DIR)

MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = Path(MODEL_ID).is_dir()

OUTPUT_ROOT = SNU_ROOT / "qwen2vl_7b_multitask_bipair_conditional_v1"
# Set this to a specific run id when needed. If None, the latest run is used.
PILOT_RUN_ID = None

def resolve_run_root(output_root, run_id=None):
    runs_root = output_root / "runs"
    if run_id is not None:
        run_root = runs_root / run_id
        assert run_root.is_dir(), run_root
        return run_root
    candidates = sorted([p for p in runs_root.iterdir() if p.is_dir()])
    if not candidates:
        raise RuntimeError(f"No runs found under {runs_root}")
    return candidates[-1]

RUN_ROOT = resolve_run_root(OUTPUT_ROOT, PILOT_RUN_ID)
RUN_ID = RUN_ROOT.name
OUTPUT_DIR = RUN_ROOT / "multitask_bipair_conditional"
EVAL_DIR = OUTPUT_DIR / "eval"
BEST_ADAPTER_DIR = OUTPUT_DIR / "best_adapter"
SUBMIT_PATH = OUTPUT_DIR / "submission_pilot_c.csv"
for path in [EVAL_DIR, BEST_ADAPTER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
VALID_RATIO = 0.10
TRAIN_ROWS = None
VALID_ROWS = None
QUICK_EVAL_ROWS = 50
QUICK_EVAL_ROWS = 50
FULL_EVAL_ROWS = 300
TOP_K_FULL_EVAL = 5
SCORE_BATCH_SIZE = 4

TASK_RATIOS = {
    "order": 0.30,
    "pairwise": 0.25,
    "first": 0.10,
    "last": 0.10,
    "fixed_first": 0.10,
    "fixed_last": 0.10,
    "fixed_endpoints": 0.05,
}
TASK_LOSS_WEIGHTS = {task: 1.0 for task in TASK_RATIOS}

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 1
MAX_TRAIN_STEPS = -1
SAVE_STEPS = 100
LOGGING_STEPS = 20

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("output:", OUTPUT_DIR)
print("model:", MODEL_ID)


In [ ]:
# 3) Data split, prompts, metrics, and model helpers
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def compact_order(order):
    return " ".join(str(int(value)) for value in order)


def parse_compact_order(text, expected_len=4):
    values = [int(x) for x in re.findall(r"[1-4]", str(text))]
    if len(values) != expected_len or len(set(values)) != expected_len:
        return None
    return values


def row_image_paths(row, image_root=TRAIN_IMAGE_DIR):
    sample_id = str(row["Id"])
    return [str(Path(image_root) / sample_id / str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def base_record(row_index, row):
    answer = [int(value) for value in row["Answer_list"]]
    order = order_to_sequence(answer)
    return {
        "row_index": int(row_index),
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order,
        "image_paths": row_image_paths(row, TRAIN_IMAGE_DIR),
    }


def pair_target_for_order(order, a, b):
    ranks = {frame: idx for idx, frame in enumerate(order)}
    return "A" if ranks[int(a)] < ranks[int(b)] else "B"


def build_record_pools(dataframe):
    pools = {task: [] for task in TASK_RATIOS}
    for row_index, row in dataframe.iterrows():
        base = base_record(row_index, row)
        order = base["order"]

        item = copy.deepcopy(base)
        item.update({"task_type": "order", "target": compact_order(order)})
        pools["order"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "first", "target": str(order[0])})
        pools["first"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "last", "target": str(order[-1])})
        pools["last"].append(item)

        for i, j in PAIR_INDICES:
            a, b = i + 1, j + 1
            for left, right in [(a, b), (b, a)]:
                item = copy.deepcopy(base)
                item.update({
                    "task_type": "pairwise",
                    "pair": [left, right],
                    "image_paths": [base["image_paths"][left - 1], base["image_paths"][right - 1]],
                    "target": pair_target_for_order(order, left, right),
                })
                pools["pairwise"].append(item)

        first = order[0]
        remaining = [x for x in order if x != first]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_first", "fixed_first": first, "target": compact_order(remaining)})
        pools["fixed_first"].append(item)

        last = order[-1]
        remaining = [x for x in order if x != last]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_last", "fixed_last": last, "target": compact_order(remaining)})
        pools["fixed_last"].append(item)

        middle = [x for x in order if x not in {order[0], order[-1]}]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_endpoints", "fixed_first": order[0], "fixed_last": order[-1], "target": compact_order(middle)})
        pools["fixed_endpoints"].append(item)
    return pools


def sample_records(records, count, rng):
    indices = rng.integers(0, len(records), size=count)
    return [records[int(index)] for index in indices]


def build_balanced_records(dataframe):
    pools = build_record_pools(dataframe)
    base_total = int(math.ceil(len(pools["order"]) / TASK_RATIOS["order"]))
    rng = np.random.default_rng(SEED)
    merged = []
    distribution = {}
    for task, ratio in TASK_RATIOS.items():
        count = max(1, int(round(base_total * ratio)))
        records = sample_records(pools[task], count, rng)
        merged.extend(records)
        distribution[task] = {"pool": len(pools[task]), "sampled": len(records)}
        print(task, distribution[task])
    rng.shuffle(merged)
    return merged, pools, distribution


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

split_root = SNU_ROOT / "id_splits" / "qwen2vl_lgt_order_refine_20260714_003635"
if (split_root / "train_ids.json").exists() and (split_root / "validation_ids.json").exists():
    train_ids = set(str(x) for x in json.load(open(split_root / "train_ids.json", "r", encoding="utf-8")))
    valid_ids = set(str(x) for x in json.load(open(split_root / "validation_ids.json", "r", encoding="utf-8")))
else:
    unique_ids = train_df["Id"].unique().copy()
    rng = np.random.default_rng(SEED)
    rng.shuffle(unique_ids)
    valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
    valid_ids = set(unique_ids[:valid_size])
    train_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(train_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)
if TRAIN_ROWS is not None:
    training_df = training_df.sample(n=min(TRAIN_ROWS, len(training_df)), random_state=SEED).reset_index(drop=True)
if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

train_records, train_pools, task_distribution = build_balanced_records(training_df)
valid_pools = build_record_pools(validation_df)
quick_eval_df = validation_df.sample(n=min(QUICK_EVAL_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

run_config = {
    "experiment": "qwen2vl_7b_multitask_bipair_conditional_v1",
    "run_id": RUN_ID,
    "model_repo_id": MODEL_REPO_ID,
    "model_id": MODEL_ID,
    "output_dir": str(OUTPUT_DIR),
    "task_ratios": TASK_RATIOS,
    "task_loss_weights": TASK_LOSS_WEIGHTS,
    "task_distribution": task_distribution,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "target_modules": LORA_TARGET_MODULES},
    "learning_rate": LEARNING_RATE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_train_steps": MAX_TRAIN_STEPS,
    "save_steps": SAVE_STEPS,
    "seed": SEED,
    "train_rows": len(training_df),
    "validation_rows": len(validation_df),
}
with open(RUN_ROOT / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "task_distribution.json", "w", encoding="utf-8") as f:
    json.dump(task_distribution, f, ensure_ascii=False, indent=2)

print("train/valid/test:", len(training_df), len(validation_df), len(test_df))
print("train records:", len(train_records))


processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def task_instruction(example):
    return globals()["task_instruction_train"](example) if "task_instruction_train" in globals() else task_instruction_impl(example)


def task_instruction_impl(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return f"Caption:\n{sentence}\n\nThe two candidate images are labeled A and B in the presented order.\nWhich image occurs earlier in the story timeline?\nAnswer only A or B."
    if task_type == "first":
        return f"Caption:\n{sentence}\n\nWhich image is the first scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "last":
        return f"Caption:\n{sentence}\n\nWhich image is the last scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "order":
        return f"Caption:\n{sentence}\n\nOrder all four images from earliest to latest in the story.\nAnswer only four frame numbers separated by spaces, for example: 1 2 3 4."
    if task_type == "fixed_first":
        return f"Caption:\n{sentence}\n\nFrame {example['fixed_first']} is fixed as the first scene.\nOrder the remaining frames from earliest to latest.\nAnswer only the remaining frame numbers separated by spaces."
    if task_type == "fixed_last":
        return f"Caption:\n{sentence}\n\nFrame {example['fixed_last']} is fixed as the last scene.\nOrder the remaining frames from earliest to latest.\nAnswer only the remaining frame numbers separated by spaces."
    if task_type == "fixed_endpoints":
        return f"Caption:\n{sentence}\n\nFrame {example['fixed_first']} is fixed as the first scene.\nFrame {example['fixed_last']} is fixed as the last scene.\nOrder the remaining middle frames from earliest to latest.\nAnswer only the remaining frame numbers separated by spaces."
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        label = "A" if example["task_type"] == "pairwise" and idx == 1 else "B" if example["task_type"] == "pairwise" and idx == 2 else str(idx)
        content.append({"type": "text", "text": f"\nImage {label}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction_impl(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


def checkpoint_name(path):
    return Path(path).name


def find_checkpoint_dirs():
    dirs = []
    for path in OUTPUT_DIR.iterdir():
        if path.name.startswith("checkpoint-") and (path / "adapter_config.json").exists():
            dirs.append(path)
    final_dir = OUTPUT_DIR / "final_adapter"
    if (final_dir / "adapter_config.json").exists():
        dirs.append(final_dir)
    def sort_key(path):
        match = re.findall(r"checkpoint-(\d+)", str(path))
        return int(match[-1]) if match else 10**9
    return sorted(dict.fromkeys(dirs), key=sort_key)


EVAL_MODEL = None
CURRENT_ADAPTER_NAME = None


def adapter_name_for(adapter_dir):
    return re.sub(r"[^0-9a-zA-Z_]+", "_", checkpoint_name(adapter_dir))


def load_eval_model(adapter_dir):
    global EVAL_MODEL, CURRENT_ADAPTER_NAME
    adapter_dir = str(adapter_dir)
    adapter_name = adapter_name_for(adapter_dir)

    def load_fresh():
        base = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
            local_files_only=MODEL_LOCAL_FILES_ONLY,
            trust_remote_code=True,
        )
        from peft import PeftModel
        return PeftModel.from_pretrained(base, adapter_dir, adapter_name=adapter_name, is_trainable=False)

    if EVAL_MODEL is None:
        EVAL_MODEL = load_fresh()
    else:
        try:
            if CURRENT_ADAPTER_NAME is not None and hasattr(EVAL_MODEL, "delete_adapter"):
                EVAL_MODEL.delete_adapter(CURRENT_ADAPTER_NAME)
            EVAL_MODEL.load_adapter(adapter_dir, adapter_name=adapter_name, is_trainable=False)
            EVAL_MODEL.set_adapter(adapter_name)
        except Exception as exc:
            print("Adapter switch failed; reloading base model:", repr(exc))
            del EVAL_MODEL
            gc.collect()
            torch.cuda.empty_cache()
            EVAL_MODEL = load_fresh()

    CURRENT_ADAPTER_NAME = adapter_name
    EVAL_MODEL.eval()
    if hasattr(EVAL_MODEL, "generation_config"):
        EVAL_MODEL.generation_config.do_sample = False
        EVAL_MODEL.generation_config.temperature = None
        EVAL_MODEL.generation_config.top_p = None
        EVAL_MODEL.generation_config.top_k = None
        EVAL_MODEL.generation_config.num_beams = 1
    return EVAL_MODEL

def model_device(active_model):
    return next(active_model.parameters()).device


def single_token_id(value):
    ids = processor.tokenizer.encode(str(value), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"{value!r} tokenized to {ids}")
    return ids[0]


DIGIT_TOKEN_IDS = {digit: single_token_id(str(digit)) for digit in [1, 2, 3, 4]}
AB_TOKEN_IDS = {"A": single_token_id("A"), "B": single_token_id("B")}


def make_eval_example(row, task_type, pair=None, fixed_first=None, fixed_last=None, image_root=TRAIN_IMAGE_DIR):
    answer = [int(value) for value in row.get("Answer_list", [1, 2, 3, 4])]
    image_paths = row_image_paths(row, image_root)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order_to_sequence(answer),
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["pair"] = [a, b]
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    if fixed_first is not None:
        example["fixed_first"] = int(fixed_first)
    if fixed_last is not None:
        example["fixed_last"] = int(fixed_last)
    return example


In [ ]:
# 4) Scoring and conditional sequential decoding
@torch.no_grad()
def score_next_token_candidates(active_model, example, token_ids):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
    outputs = active_model(**inputs)
    last_pos = int(inputs["attention_mask"][0].sum().item()) - 1
    logits = outputs.logits[0, last_pos]
    ids = list(token_ids.values())
    probs = torch.softmax(logits[ids].float(), dim=-1).detach().cpu().numpy()
    processor.tokenizer.padding_side = old_padding_side
    return {key: float(prob) for key, prob in zip(token_ids.keys(), probs)}


@torch.no_grad()
def generate_text(active_model, example, max_new_tokens=16):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
    generated = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    output = processor.tokenizer.batch_decode(generated[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    processor.tokenizer.padding_side = old_padding_side
    return output.strip()


@torch.no_grad()
def score_sequence_candidates(active_model, example, candidate_orders, batch_size=None):
    batch_size = int(batch_size or SCORE_BATCH_SIZE)
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    prompt = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
    prompt_len = int(prompt_inputs["attention_mask"][0].sum().item())
    scores = {}
    orders = list(candidate_orders)
    for start_index in range(0, len(orders), batch_size):
        batch_orders = orders[start_index:start_index + batch_size]
        texts = [prompt + compact_order(order) for order in batch_orders]
        batch_images = [images for _ in batch_orders]
        inputs = processor(text=texts, images=batch_images, padding=True, return_tensors="pt")
        inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = active_model(**inputs)
        for row_index, order in enumerate(batch_orders):
            input_ids = inputs["input_ids"][row_index]
            attention_len = int(inputs["attention_mask"][row_index].sum().item())
            target_len = attention_len - prompt_len
            target_ids = input_ids[prompt_len:prompt_len + target_len]
            logits = outputs.logits[row_index, prompt_len - 1:prompt_len - 1 + target_len]
            log_probs = torch.log_softmax(logits.float(), dim=-1)
            scores[" ".join(map(str, order))] = float(log_probs.gather(1, target_ids[:, None]).mean().item())
    processor.tokenizer.padding_side = old_padding_side
    return scores

def softmax_scores(scores, temperature=1.0):
    keys = list(scores.keys())
    values = np.array([scores[k] for k in keys], dtype=np.float64) / temperature
    values = values - values.max()
    probs = np.exp(values)
    probs = probs / probs.sum()
    return {k: float(v) for k, v in zip(keys, probs)}


def parse_order_key(key):
    return tuple(int(x) for x in str(key).split())


def pair_logit(p, eps=1e-6):
    p = min(max(float(p), eps), 1.0 - eps)
    return math.log(p / (1.0 - p))


def combine_bidirectional_pair(p_forward, p_reverse, eps=1e-6):
    combined_logit = 0.5 * (pair_logit(p_forward, eps) - pair_logit(p_reverse, eps))
    return 1.0 / (1.0 + math.exp(-combined_logit))


def normalized(scores):
    total = sum(max(float(v), 0.0) for v in scores.values())
    if total <= 0:
        return {int(k): 1.0 / len(scores) for k in scores}
    return {int(k): max(float(v), 0.0) / total for k, v in scores.items()}


def position_marginal(order_probs, position, candidates):
    out = {int(c): 0.0 for c in candidates}
    for key, prob in order_probs.items():
        order = parse_order_key(key)
        if order[position] in out:
            out[order[position]] += float(prob)
    return normalized(out)


def endpoint_probs_from_pair(pair_probs, candidates, mode):
    candidates = [int(x) for x in candidates]
    if mode == "first":
        return normalized({i: sum(pair_probs[f"{i}>{j}"]["combined_prob"] for j in candidates if j != i) for i in candidates})
    return normalized({i: sum(pair_probs[f"{j}>{i}"]["combined_prob"] for j in candidates if j != i) for i in candidates})


def fuse_three(order_signal, endpoint_signal, pair_signal, w_order, w_endpoint, w_pair):
    candidates = sorted(set(order_signal) | set(endpoint_signal) | set(pair_signal))
    return normalized({i: w_order * order_signal.get(i, 0.0) + w_endpoint * endpoint_signal.get(i, 0.0) + w_pair * pair_signal.get(i, 0.0) for i in candidates})


def top_with_margin(scores):
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return int(ranked[0][0]), float(ranked[0][1] - (ranked[1][1] if len(ranked) > 1 else 0.0))


def score_sample(active_model, row, image_root=TRAIN_IMAGE_DIR, has_gold=True):
    gold_order = order_to_sequence(row["Answer_list"]) if has_gold else None
    sample_id = str(row["Id"])
    direct_text = generate_text(active_model, make_eval_example(row, "order", image_root=image_root))
    direct_order = parse_compact_order(direct_text, 4)

    first_probs = score_next_token_candidates(active_model, make_eval_example(row, "first", image_root=image_root), DIGIT_TOKEN_IDS)
    last_probs = score_next_token_candidates(active_model, make_eval_example(row, "last", image_root=image_root), DIGIT_TOKEN_IDS)
    first_probs = {str(k): v for k, v in first_probs.items()}
    last_probs = {str(k): v for k, v in last_probs.items()}

    pair_probs = {}
    for i, j in PAIR_INDICES:
        a, b = i + 1, j + 1
        forward = score_next_token_candidates(active_model, make_eval_example(row, "pairwise", pair=(a, b), image_root=image_root), AB_TOKEN_IDS)
        reverse = score_next_token_candidates(active_model, make_eval_example(row, "pairwise", pair=(b, a), image_root=image_root), AB_TOKEN_IDS)
        p_forward = forward["A"]
        p_reverse = reverse["A"]
        combined = combine_bidirectional_pair(p_forward, p_reverse)
        pair_probs[f"{a}>{b}"] = {"forward_prob": float(p_forward), "reverse_prob": float(p_reverse), "combined_prob": float(combined), "swap_inconsistency": float(abs(p_forward + p_reverse - 1.0))}
        pair_probs[f"{b}>{a}"] = {"forward_prob": float(1.0 - p_forward), "reverse_prob": float(1.0 - p_reverse), "combined_prob": float(1.0 - combined), "swap_inconsistency": float(abs(p_forward + p_reverse - 1.0))}

    order_scores = score_sequence_candidates(active_model, make_eval_example(row, "order", image_root=image_root), PERMUTATIONS)
    order_probs = softmax_scores(order_scores)

    return {
        "sample_id": sample_id,
        "gold_order": gold_order,
        "direct_order": direct_order,
        "direct_text": direct_text,
        "first_probs": first_probs,
        "last_probs": last_probs,
        "bidirectional_pair_probs": pair_probs,
        "order_24_scores": order_scores,
        "order_24_probs": order_probs,
    }


def conditional_decode(active_model, row, scored, image_root=TRAIN_IMAGE_DIR):
    frames = [1, 2, 3, 4]
    order_probs = scored["order_24_probs"]
    pair_probs = scored["bidirectional_pair_probs"]
    order_first = position_marginal(order_probs, 0, frames)
    order_last = position_marginal(order_probs, 3, frames)
    first_head = normalized({int(k): v for k, v in scored["first_probs"].items()})
    last_head = normalized({int(k): v for k, v in scored["last_probs"].items()})
    pair_first = endpoint_probs_from_pair(pair_probs, frames, "first")
    pair_last = endpoint_probs_from_pair(pair_probs, frames, "last")
    fused_first = fuse_three(order_first, first_head, pair_first, 0.35, 0.25, 0.40)
    fused_last = fuse_three(order_last, last_head, pair_last, 0.35, 0.25, 0.40)
    first_candidate, first_margin = top_with_margin(fused_first)
    last_candidate, last_margin = top_with_margin(fused_last)

    if first_margin >= last_margin:
        first = first_candidate
        remaining = [x for x in frames if x != first]
        candidates = list(itertools.permutations(remaining))
        cond_scores = score_sequence_candidates(active_model, make_eval_example(row, "fixed_first", fixed_first=first, image_root=image_root), candidates)
        cond_probs = softmax_scores(cond_scores)
        conditional_order_last = position_marginal(cond_probs, -1, remaining)
        conditional_fused = fuse_three(conditional_order_last, normalized({i: float(scored["last_probs"][str(i)]) for i in remaining}), endpoint_probs_from_pair(pair_probs, remaining, "last"), 0.40, 0.20, 0.40)
        last = max(conditional_fused, key=conditional_fused.get)
        fixed_first_selected = True
    else:
        last = last_candidate
        remaining = [x for x in frames if x != last]
        candidates = list(itertools.permutations(remaining))
        cond_scores = score_sequence_candidates(active_model, make_eval_example(row, "fixed_last", fixed_last=last, image_root=image_root), candidates)
        cond_probs = softmax_scores(cond_scores)
        conditional_order_first = position_marginal(cond_probs, 0, remaining)
        conditional_fused = fuse_three(conditional_order_first, normalized({i: float(scored["first_probs"][str(i)]) for i in remaining}), endpoint_probs_from_pair(pair_probs, remaining, "first"), 0.40, 0.20, 0.40)
        first = max(conditional_fused, key=conditional_fused.get)
        fixed_first_selected = False

    middle = [x for x in frames if x not in {first, last}]
    middle_candidates = list(itertools.permutations(middle))
    middle_scores = score_sequence_candidates(active_model, make_eval_example(row, "fixed_endpoints", fixed_first=first, fixed_last=last, image_root=image_root), middle_candidates)
    middle_probs = softmax_scores(middle_scores)
    a, b = middle
    order_a_before_b = middle_probs.get(f"{a} {b}", 0.0)
    pair_a_before_b = pair_probs[f"{a}>{b}"]["combined_prob"]
    score_a_before_b = 0.5 * order_a_before_b + 0.5 * pair_a_before_b
    middle_order = [a, b] if score_a_before_b >= 0.5 else [b, a]
    final_order = [first, *middle_order, last]
    scored.update({
        "fused_first": fused_first,
        "fused_last": fused_last,
        "first_margin": first_margin,
        "last_margin": last_margin,
        "first_endpoint_selected": fixed_first_selected,
        "fixed_endpoint": first if fixed_first_selected else last,
        "selected_other_endpoint": last if fixed_first_selected else first,
        "conditional_order_6_probs": cond_probs,
        "conditional_middle_probs": middle_probs,
        "final_order": final_order,
    })
    return scored


def order_metric_row(pred_order, gold_order):
    if pred_order is None or gold_order is None:
        return {"exact": 0.0, "first": 0.0, "last": 0.0, "both_endpoints": 0.0, "position": 0.0, "relative_pair": 0.0}
    ranks_p = {x: i for i, x in enumerate(pred_order)}
    ranks_g = {x: i for i, x in enumerate(gold_order)}
    return {
        "exact": float(pred_order == gold_order),
        "first": float(pred_order[0] == gold_order[0]),
        "last": float(pred_order[-1] == gold_order[-1]),
        "both_endpoints": float(pred_order[0] == gold_order[0] and pred_order[-1] == gold_order[-1]),
        "position": float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
        "relative_pair": float(np.mean([(ranks_p[a] < ranks_p[b]) == (ranks_g[a] < ranks_g[b]) for a, b in itertools.combinations([1, 2, 3, 4], 2)])),
    }


In [ ]:
# 5) Quick eval, full eval, best checkpoint selection, and test inference
def evaluate_checkpoint(adapter_dir, rows, tag, image_root=TRAIN_IMAGE_DIR, has_gold=True):
    ckpt = checkpoint_name(adapter_dir)
    pred_dir = EVAL_DIR / "sample_predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    cache_path = pred_dir / f"{ckpt}_{tag}.json"
    if cache_path.exists():
        with open(cache_path, "r", encoding="utf-8") as f:
            records = json.load(f)
    else:
        active_model = load_eval_model(adapter_dir)
        records = []
        for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"{ckpt} {tag}"):
            scored = score_sample(active_model, row, image_root=image_root, has_gold=has_gold)
            scored = conditional_decode(active_model, row, scored, image_root=image_root)
            records.append(scored)
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False, indent=2)

    if not has_gold:
        return {}, records
    metric_rows = []
    for r in records:
        direct = order_metric_row(r["direct_order"], r["gold_order"])
        final = order_metric_row(r["final_order"], r["gold_order"])
        metric_rows.append({
            "sample_id": r["sample_id"],
            "direct_exact": direct["exact"],
            "direct_position": direct["position"],
            "conditional_sequential_exact": final["exact"],
            "conditional_sequential_first": final["first"],
            "conditional_sequential_last": final["last"],
            "conditional_sequential_both_endpoints": final["both_endpoints"],
            "conditional_sequential_position": final["position"],
            "conditional_sequential_relative_pair": final["relative_pair"],
            "bidirectional_pair_accuracy": float(np.mean([
                ((r["bidirectional_pair_probs"][f"{a}>{b}"]["combined_prob"] >= 0.5) == ({x: i for i, x in enumerate(r["gold_order"])}[a] < {x: i for i, x in enumerate(r["gold_order"])}[b]))
                for a, b in itertools.combinations([1, 2, 3, 4], 2)
            ])),
            "mean_swap_inconsistency": float(np.mean([v["swap_inconsistency"] for k, v in r["bidirectional_pair_probs"].items() if k < k[::-1]])),
        })
    df = pd.DataFrame(metric_rows)
    summary = {col: float(df[col].mean()) for col in df.columns if col != "sample_id"}
    summary.update({"checkpoint": ckpt, "tag": tag, "adapter_dir": str(adapter_dir), "rows": len(rows)})
    return summary, records


checkpoint_dirs = find_checkpoint_dirs()
print("checkpoints:", [checkpoint_name(path) for path in checkpoint_dirs])
quick_rows = validation_df.sample(n=min(QUICK_EVAL_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)
quick_summaries = []
for adapter_dir in checkpoint_dirs:
    summary, _ = evaluate_checkpoint(adapter_dir, quick_rows, f"quick{len(quick_rows)}")
    quick_summaries.append(summary)
quick_df = pd.DataFrame(quick_summaries).sort_values(
    ["conditional_sequential_exact", "direct_exact", "conditional_sequential_both_endpoints", "bidirectional_pair_accuracy", "conditional_sequential_relative_pair"],
    ascending=False,
).reset_index(drop=True)
quick_df.to_csv(EVAL_DIR / "checkpoint_metrics_quick.csv", index=False)
display(quick_df)

top_names = quick_df.head(TOP_K_FULL_EVAL)["checkpoint"].tolist()
name_to_dir = {checkpoint_name(path): path for path in checkpoint_dirs}
full_rows = validation_df.sample(n=min(FULL_EVAL_ROWS, len(validation_df)), random_state=SEED + 1).reset_index(drop=True)
full_summaries = []
for ckpt in top_names:
    summary, _ = evaluate_checkpoint(name_to_dir[ckpt], full_rows, f"full{len(full_rows)}")
    full_summaries.append(summary)
full_df = pd.DataFrame(full_summaries).sort_values(
    ["conditional_sequential_exact", "direct_exact", "conditional_sequential_both_endpoints", "bidirectional_pair_accuracy", "conditional_sequential_relative_pair"],
    ascending=False,
).reset_index(drop=True)
full_df.to_csv(EVAL_DIR / "checkpoint_metrics_full.csv", index=False)
display(full_df)

best = full_df.iloc[0].to_dict()
best_adapter_dir = Path(best["adapter_dir"])
if BEST_ADAPTER_DIR.exists():
    shutil.rmtree(BEST_ADAPTER_DIR)
shutil.copytree(best_adapter_dir, BEST_ADAPTER_DIR)
processor.save_pretrained(BEST_ADAPTER_DIR)

best_config = {
    "selected_from": "full_validation",
    "best": best,
    "selection_priority": ["conditional_sequential_exact", "direct_exact", "conditional_sequential_both_endpoints", "bidirectional_pair_accuracy", "conditional_sequential_relative_pair"],
}
with open(BEST_ADAPTER_DIR / "best_checkpoint.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)
shutil.copy2(EVAL_DIR / "checkpoint_metrics_full.csv", BEST_ADAPTER_DIR / "checkpoint_metrics_full.csv")
print("best adapter saved:", BEST_ADAPTER_DIR)


def sequence_to_answer(order):
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


test_records_summary, test_records = evaluate_checkpoint(BEST_ADAPTER_DIR, test_df, "test", image_root=TEST_IMAGE_DIR, has_gold=False)
submission = pd.DataFrame([{"Id": r["sample_id"], "Answer": str(sequence_to_answer(r["final_order"]))} for r in test_records])
submission.to_csv(SUBMIT_PATH, index=False)
shutil.copy2(SUBMIT_PATH, BEST_ADAPTER_DIR / "submission.csv")
display(submission.head())
print("submission saved:", SUBMIT_PATH)
